# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hajergafsi/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [7]:
print("I'm doing Lane 2: Refresh / Content Opportunity Scoring.\n\nI chose this lane because it produces a complete, concrete deliverable — a ranked list\nof pages a content reviewer should look at first, with reasons attached — and it has a\nworking starter pipeline I can learn from before extending it myself. I'm new to applying\nML in practice (though I've studied ML/DL theory), so having a real, runnable example to\nbuild on and improve is the right level of challenge for me right now.")

I'm doing Lane 2: Refresh / Content Opportunity Scoring.

I chose this lane because it produces a complete, concrete deliverable — a ranked list
of pages a content reviewer should look at first, with reasons attached — and it has a
working starter pipeline I can learn from before extending it myself. I'm new to applying
ML in practice (though I've studied ML/DL theory), so having a real, runnable example to
build on and improve is the right level of challenge for me right now.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [10]:
print("Decision this improves: Which pages should a content reviewer review and refresh first,\ngiven they can only review a limited number of pages per week?\n\nWho acts on it: A content editor/reviewer at one of FlyRank's clients. They take the top\nof my ranked list and manually decide, page by page, whether to refresh, expand, rewrite,\nor leave each one — my output guides where they look first, not what they do to the page.\n\nCost of a wrong call:\n- False positive (I flag a page that didn't need attention): wasted reviewer time.\n- False negative (I miss a page that really was declining or under-optimized): a real\n  opportunity or problem goes unnoticed, possibly for a long time.\nSince reviewer time is limited, precision@K (are my top picks actually good ones?) matters\nmore than a generic accuracy score — it matches how the ranked list will actually be used.\n\nWhy ML instead of just a rule: A simple rule (e.g. \"no update in 180 days AND 500+\nimpressions\") is a reasonable start, and I'll build one as my baseline. But real\n\"worth reviewing\" pages likely depend on several signals interacting (visibility, position,\nCTR, freshness, engagement) in ways that are hard to hand-write as one clean if-statement.\nThat's exactly where I'll test whether a trained model earns its extra complexity over\nthe simple rule.");

Decision this improves: Which pages should a content reviewer review and refresh first,
given they can only review a limited number of pages per week?

Who acts on it: A content editor/reviewer at one of FlyRank's clients. They take the top
of my ranked list and manually decide, page by page, whether to refresh, expand, rewrite,
or leave each one — my output guides where they look first, not what they do to the page.

Cost of a wrong call:
- False positive (I flag a page that didn't need attention): wasted reviewer time.
- False negative (I miss a page that really was declining or under-optimized): a real
  opportunity or problem goes unnoticed, possibly for a long time.
Since reviewer time is limited, precision@K (are my top picks actually good ones?) matters
more than a generic accuracy score — it matches how the ranked list will actually be used.

Why ML instead of just a rule: A simple rule (e.g. "no update in 180 days AND 500+
impressions") is a reasonable start, and I'll build on

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [14]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total rows:", len(df))
print("Unique clients:", df["client_id"].nunique())

# Respect the data gotchas: avg_position == 0 means "no data", not rank zero
valid_position = df[df["avg_position"] > 0]
print("Rows with valid position data:", len(valid_position))

# trend_direction is the source of the label — never a feature, just look at its spread here
print("\nTrend direction breakdown:")
print(df["trend_direction"].value_counts(normalize=True).round(3) * 100)

# A quick look at how many pages are both visible (real impressions) and stale
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print("\nStale but still visible pages:", len(stale_visible),
      f"({len(stale_visible)/len(df):.1%} of all pages)")

Total rows: 30000
Unique clients: 32
Rows with valid position data: 28795

Trend direction breakdown:
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64

Stale but still visible pages: 17 (0.1% of all pages)


In [16]:
print("54.2% of pages are declining, and 17 pages are both stale\n(180+ days since update) and still getting real search visibility (500+ impressions) —\nthese are strong early candidates for review, which is exactly what this lane is meant\nto surface.")

54.2% of pages are declining, and 17 pages are both stale
(180+ days since update) and still getting real search visibility (500+ impressions) —
these are strong early candidates for review, which is exactly what this lane is meant
to surface.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [17]:
print("What I can claim:\n- Observed patterns: e.g. \"pages with X characteristic are more likely to show the\n  declining-trend label in this snapshot.\"\n- Directional/decision-support results: \"this ranked list highlights pages worth a\n  reviewer's limited time first, based on the evidence available.\"\n- Comparative claims: \"the model's top-50 picks matched the trend-decline label more\n  often than the simple rule-based baseline did, on this data.\"\n\nWhat I will NOT claim:\n- That refreshing a flagged page will cause traffic to recover — that requires an\n  actual experiment (e.g. A/B test), which I'm not running here.\n- Anything about how Google's search algorithm actually works.\n- That my proxy label (current trend_direction) is a perfect measure of \"needs review\" —\n  it's a simplification I'm using to get started, not ground truth.\n- Any claim built on trend_direction/trend_pct as a *feature* — those columns define\n  my label, so using them as inputs would just teach the model to copy the label\n  (a circular result), not discover anything real.")

What I can claim:
- Observed patterns: e.g. "pages with X characteristic are more likely to show the
  declining-trend label in this snapshot."
- Directional/decision-support results: "this ranked list highlights pages worth a
  reviewer's limited time first, based on the evidence available."
- Comparative claims: "the model's top-50 picks matched the trend-decline label more
  often than the simple rule-based baseline did, on this data."

What I will NOT claim:
- That refreshing a flagged page will cause traffic to recover — that requires an
  actual experiment (e.g. A/B test), which I'm not running here.
- Anything about how Google's search algorithm actually works.
- That my proxy label (current trend_direction) is a perfect measure of "needs review" —
  it's a simplification I'm using to get started, not ground truth.
- Any claim built on trend_direction/trend_pct as a *feature* — those columns define
  my label, so using them as inputs would just teach the model to copy the label


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.